<a href="https://colab.research.google.com/github/NeonsCandy/HW14.ipynb/blob/main/HW14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Bidirectional, Dense
import matplotlib.pyplot as plt

max_features = 10000
maxlen = 500
batch_size = 128
epochs = 5

print("Завантаження даних...")
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

print("Вирівнювання послідовностей...")
x_train = pad_sequences(x_train, maxlen=maxlen)
x_test = pad_sequences(x_test, maxlen=maxlen)
print(f"Формат тренувальних даних: {x_train.shape}")
print(f"Формат тестових даних: {x_test.shape}")

#Simple RNN
model_rnn = Sequential([
    Embedding(max_features, 32),
    SimpleRNN(32),
    Dense(1, activation='sigmoid')
])

model_rnn.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
print("Навчання Simple RNN...")
history_rnn = model_rnn.fit(x_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.2)

#LSTM
model_lstm = Sequential([
    Embedding(max_features, 32),
    LSTM(32),
    Dense(1, activation='sigmoid')
])

model_lstm.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
print("Навчання LSTM...")
history_lstm = model_lstm.fit(x_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.2)

# Bidirectional LSTM
model_bidir = Sequential([
    Embedding(max_features, 32),
    Bidirectional(LSTM(32)),
    Dense(1, activation='sigmoid')
])

model_bidir.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
print("Навчання Bidirectional LSTM...")
history_bidir = model_bidir.fit(x_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.2)

#Deep LSTM
model_deep = Sequential([
    Embedding(max_features, 32),
    LSTM(32, return_sequences=True),
    LSTM(32),
    Dense(1, activation='sigmoid')
])

model_deep.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
print("Навчання Deep LSTM...")
history_deep = model_deep.fit(x_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.2)

results = {
    "Simple RNN": model_rnn.evaluate(x_test, y_test, verbose=0)[1],
    "LSTM": model_lstm.evaluate(x_test, y_test, verbose=0)[1],
    "Bidirectional LSTM": model_bidir.evaluate(x_test, y_test, verbose=0)[1],
    "Deep LSTM": model_deep.evaluate(x_test, y_test, verbose=0)[1]
}

print("\n ПІДСУМКОВА ТОЧНІСТЬ НА ТЕСТОВИХ ДАНИХ")
for name, acc in results.items():
    print(f"{name}: {acc * 100:.2f}%")

Завантаження даних...
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Вирівнювання послідовностей...
Формат тренувальних даних: (25000, 500)
Формат тестових даних: (25000, 500)
Навчання Simple RNN...
Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 40s 240ms/step - accuracy: 0.6990 - loss: 0.5680 - val_accuracy: 0.7708 - val_loss: 0.4920
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 38s 224ms/step - accuracy: 0.8328 - loss: 0.3915 - val_accuracy: 0.7718 - val_loss: 0.4726
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 35s 220ms/step - accuracy: 0.8741 - loss: 0.3125 - val_accuracy: 0.8560 - val_loss: 0.3950
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 34s 217ms/step - accuracy: 0.9028 - loss: 0.2510 - val_accuracy: 0.8556 - val_loss: 0.3619
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 33s 207ms/step - accuracy: 0.9334 - loss: 0.1847 - val_accuracy: 0.8118 - val_loss: 0.5065
Навчання LSTM...
Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 61s 374ms/step - accuracy: 0.6216 - loss: 0.6341 - val_accuracy: 0.7188 - val_loss: 0.5422


Висновки:

1. Simple RNN: Показала найнижчу точність та швидко перенавчалася. Це пов'язано з тим, що прості RNN не здатне утримувати довгострокову пам'ять через проблему згасаючого градієнта — до кінця рецензії з 500 слів мережа "забуває", про що йшлося на початку.

2. LSTM: Значно перевершила просту RNN . Завдяки архітектурі комірок пам'яті (forget, input, output gates), модель успішно виявляє довгострокові залежності у тексті та краще визначає загальну тональність відгуку.

3. Bidirectional LSTM: Показала найкращий або один із найкращих результатів. Здатність читати контекст як у прямому, так і в зворотному напрямку є критично важливою для розуміння сарказму, заперечень або складних конструкцій у рецензіях.

4. Deep LSTM: Хоча глибока модель має більше параметрів і здатна виявляти більш абстрактні ознаки, на такому відносно простому датасеті як IMDB вона часто не дає значного приросту точності порівняно зі звичайною LSTM. Більше того тренується вдвічі довше і має вищу схильність до швидкого перенавчан.